In [12]:
from pathlib import Path
import pandas as pd

streams = pd.read_parquet("../data/processed/streams_enriched.parquet")
OUT = Path("../data/processed/extracts")
OUT.mkdir(parents=True, exist_ok=True)

In [13]:
monthly_genre = (
    streams.groupby(["month", "genre"])
    .agg(streams_n=("ts", "size"), listens=("is_listen", "sum"), ms=("ms_played", "sum"))
    .reset_index()
)
monthly_genre["hours"] = monthly_genre["ms"] / 3.6e6
monthly_genre = monthly_genre.drop(columns="ms")
monthly_genre.to_csv(OUT / "monthly_genre.csv", index=False)

In [14]:
hours_dow_genre = (
    streams.groupby(['hour', "dow", "genre"])
    .agg(streams_n=("ts", "size"), listens=("is_listen", "sum"), ms=("ms_played", "sum"))
    .reset_index()
)

hours_dow_genre["hours"] = hours_dow_genre["ms"] / 3.6e6
hours_dow_genre = hours_dow_genre.drop(columns="ms")
hours_dow_genre.to_csv(OUT / "hours_dow_genre.csv", index=False)

In [15]:
artist_year = (
    streams.groupby(["year", "artist", "genre"])
    .agg(streams_n=("ts", "size"), listens=("is_listen", "sum"), ms=("ms_played", "sum"))
    .reset_index()
)

artist_year["hours"] = artist_year["ms"] / 3.6e6
artist_year = artist_year.drop(columns="ms")
artist_year.to_csv(OUT / "artist_year.csv", index=False)

In [16]:
sessions = (
    streams.groupby("session_id")
    .agg(start=("ts", "min"), streams_n=("ts", "size"), ms=("ms_played", "sum"))
    .reset_index(drop=True)
)
sessions["minutes"] = sessions["ms"] / 60000
sessions["date"] = sessions["start"].dt.date
sessions["hour"] = sessions["start"].dt.hour
sessions.drop(columns=["start", "ms"]).to_csv(OUT / "sessions.csv", index=False)

In [17]:
first_seen = streams.groupby("artist")["ts"].min().dt.to_period("M").dt.to_timestamp()
first_seen.value_counts().sort_index().rename_axis("month").reset_index(name="new_artists") \
    .to_csv(OUT / "new_artists_monthly.csv", index=False)

pd.DataFrame([{
    "total_streams": len(streams),
    "total_hours": round(streams["ms_played"].sum() / 3.6e6),
    "artists": streams["artist"].nunique(),
    "tracks": streams["track"].nunique(),
    "first_stream": streams["ts"].min().date(),
    "last_stream": streams["ts"].max().date(),
}]).to_csv(OUT / "summary.csv", index=False)

/var/folders/27/_tq5d72j46145d36gzkszr800000gn/T/ipykernel_91878/4177053524.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  first_seen = streams.groupby("artist")["ts"].min().dt.to_period("M").dt.to_timestamp()


In [18]:
for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:28} {len(pd.read_csv(f)):>6} rows")

artist_year.csv                5727 rows
hours_dow_genre.csv            1364 rows
monthly_genre.csv               544 rows
new_artists_monthly.csv          56 rows
sessions.csv                   3158 rows
summary.csv                       1 rows


In [19]:
print(pd.DataFrame([{
    "total_streams": len(streams),
    "total_hours": round(streams["ms_played"].sum() / 3.6e6),
    "artists": streams["artist"].nunique(),
    "tracks": streams["track"].nunique(),
    "first_stream": streams["ts"].min().date(),
    "last_stream": streams["ts"].max().date(),
}]))

   total_streams  total_hours  artists  tracks first_stream last_stream
0          57200         1437     3341    8408   2021-12-02  2026-07-31
